# 02 - OOP and Inheritance (Java)

This notebook builds the same `Motor` / `TalonMotor` / `SparkMotor` hierarchy from `concept.md`, idiomatically in Java. Read `concept.md` first, especially the polymorphism section. Java, like Python, hides the pointer/reference bookkeeping C++ needs: every object variable in Java is already a reference under the hood.

## The Base Class

Java has `abstract` as an actual keyword: an `abstract` class can declare `abstract` methods with no body at all, which every concrete (non-abstract) subclass is *required* to implement, or the code won't compile.

In [1]:
abstract class Motor {
    protected String name;
    protected double currentPower;

    public Motor(String name) {
        this.name = name;
        this.currentPower = 0.0;
    }

    public abstract void setPower(double power);

    public String describe() {
        return String.format("%s: %.2f power", name, currentPower);
    }
}

`setPower` is declared `abstract` — no body, just a signature — so every subclass must supply its own. `describe` is a normal method with a real, inheritable implementation that subclasses can either use as-is or override.

`name` and `currentPower` are declared `protected`, not `private`: `TalonMotor` and `SparkMotor` need direct access to `currentPower` to implement their own `setPower`, but code outside the `Motor` hierarchy should go through `setPower`/`describe` instead of reaching in and changing it directly. See `concept.md`'s "Encapsulation and access modifiers" section for the full public/private/protected distinction.

## Subclasses That Override Behavior

`TalonMotor` and `SparkMotor` both `extends Motor`. Each implements `setPower` with its own clamp range (`@Override` isn't strictly required by the compiler, but it's a strong convention — it makes the compiler double-check that you're actually overriding something that exists on the base class, catching typos).

In [2]:
class TalonMotor extends Motor {
    static final double MAX_POWER = 1.0;

    public TalonMotor(String name) {
        super(name);
    }

    @Override
    public void setPower(double power) {
        currentPower = Math.max(-MAX_POWER, Math.min(MAX_POWER, power));
    }

    @Override
    public String describe() {
        return "[Talon] " + super.describe();
    }
}

class SparkMotor extends Motor {
    static final double MAX_POWER = 0.8; // this particular Spark has a lower safe limit

    public SparkMotor(String name) {
        super(name);
    }

    @Override
    public void setPower(double power) {
        currentPower = Math.max(-MAX_POWER, Math.min(MAX_POWER, power));
    }

    @Override
    public String describe() {
        return "[Spark] " + super.describe();
    }
}

`super(name)` calls the base class's constructor to do the shared setup; `super.describe()` calls the base class's version of `describe` from inside the override, same idea as Python's `super().describe()`. Note that `Motor motor = new Motor("test");` would fail to compile at all — you cannot instantiate an abstract class directly, only its concrete subclasses.

## Polymorphism

An array declared as `Motor[]` can hold `TalonMotor` and `SparkMotor` objects, because both *are* `Motor`s. Code that loops over it only ever refers to `Motor`'s interface (`setPower`, `describe`), yet each object still runs its own overridden version.

In [3]:
Motor[] motors = { new TalonMotor("Left Drive"), new SparkMotor("Intake Roller") };

for (Motor motor : motors) {
    motor.setPower(1.5); // deliberately out of range for both
    System.out.println(motor.describe());
}

[Talon] Left Drive: 1.00 power


[Spark] Intake Roller: 0.80 power


Every element in `motors` is declared as type `Motor`, but `motor.setPower(1.5)` still runs `TalonMotor`'s clamp for the first element and `SparkMotor`'s clamp for the second. This is dynamic dispatch, resolved automatically because in Java every object variable is a reference to the real object, never a sliced copy of it.

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor or another student.

1. Add a third subclass, `VictorMotor`, with its own `MAX_POWER` and its own `describe` override, and add an instance of it to the `motors` array.
2. Add a new abstract method to `Motor` called `stop()` that every subclass must implement (perhaps just calling `setPower(0)`), and implement it in all three subclasses.
3. Try compiling `Motor m = new Motor("test");` in a new cell and read the compiler error. What does it tell you about *why* `Motor` can't be instantiated on its own?

In [4]:
// Your code here
